# TrendSetter Agent
> 한국 유튜브 Shorts 트렌드 분석 & 캠페인 기회 탐지

**과목**: 경영 의사결정을 위한 생성형 AI | **마감**: 2026-07-01

---

## 1. 환경 설정

In [22]:
# 필요 패키지 설치 (최초 1회만 실행)
!pip install -q langchain-google-genai langchain-openai langchain-community langchain \
    google-api-python-client youtube-transcript-api pydantic diskcache python-dotenv

In [3]:
import os, getpass

os.environ['YOUTUBE_API_KEY'] = getpass.getpass('Enter YouTube API Key: ')
os.environ['GOOGLE_API_KEY']  = getpass.getpass('Enter Google(Gemini) API Key: ')
os.environ['OPENAI_API_KEY']  = getpass.getpass('Enter OPENAI API Key: ')

Enter YouTube API Key:  ········
Enter Google(Gemini) API Key:  ········
Enter OPENAI API Key:  ········


In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

yt_key     = os.getenv("YOUTUBE_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

print("YouTube API Key:", "설정됨" if yt_key     else "없음 — .env 확인 필요")
print("Google API Key :", "설정됨" if google_key else "없음 — .env 확인 필요")
print("OpenAI API Key :", "설정됨" if openai_key else "없음 — OpenAI 사용 시 .env에 추가 필요")

YouTube API Key: 설정됨
Google API Key : 설정됨
OpenAI API Key : 설정됨


---
## 2. 설정 (카테고리 키워드 & 시스템 프롬프트)

In [2]:
# 카테고리별 검색 키워드
# [v0.6] 한국 콘텐츠 비중을 높이기 위해 '한국어 고유 표현' 위주로 보강한다.
#   - _fetch_shorts는 앞 3개만 사용([:3])하므로, 한국색이 강한 키워드를 앞쪽에 배치한다.
#   - [v0.6.1] 뷰티: "겟레디윗미(GRWM)"는 한국에서 패션/OOTD 콘텐츠가 압도적이라
#     뷰티 결과를 패션으로 오염시킴 → 제거하고 메이크업/스킨케어 전용 키워드로 교체.
CATEGORY_KEYWORDS = {
    "뷰티":   ["데일리 메이크업", "국룰 메이크업", "화장법", "스킨케어 루틴", "립 메이크업", "올리브영 추천"],
    "식음료": ["먹방", "맛집 추천", "편의점 꿀조합", "집밥 레시피", "카페 추천", "요리"],
    "패션":   ["데일리룩", "코디 추천", "무신사 하울", "룩북", "패션 꿀팁", "신발 추천"],
    "테크":   ["아이폰 꿀팁", "가성비 노트북", "앱 추천", "갤럭시 팁", "가젯 리뷰"],
    "피트니스":["홈트", "다이어트 운동", "헬스 루틴", "요가 스트레칭", "필라테스"],
}

SYSTEM_PROMPT = """당신은 브랜드 마케터 전용 유튜브 트렌드 분석 AI입니다.

역할:
- 한국 유튜브 Shorts 트렌드를 수집·분석하여 브랜드 캠페인 기회를 발굴합니다
- 마케터가 5분 안에 캠페인 방향을 결정할 수 있는 정보를 제공합니다

출력 형식:
- 트렌드 카드 3~5개: 제목 / 카테고리(챌린지·밈·리뷰·이슈) / 한 줄 요약 / 브랜드 기회 점수(1~5) / 캠페인 방향
- 마지막에 [골든타임 트렌드] 1개 추천 + 이유

브랜드 기회 점수 기준:
1점: 브랜드와 무관 | 2점: 간접 언급 | 3점: 소셜 테스트 | 4점: 콘텐츠 협업 | 5점: 즉시 캠페인 런칭

규칙:
- 반드시 한국어로 응답하세요
- 이전 대화의 트렌드 카드를 기억하고 후속 질문에 활용하세요
- 데이터 없이 추측으로 점수를 주지 마세요
"""

print("설정 완료")

설정 완료


---
## 3. Pydantic 스키마 (Structured Output)

In [3]:
from pydantic import BaseModel, Field

class TrendCard(BaseModel):
    title: str          = Field(description="트렌드 제목")
    category: str       = Field(description="챌린지/밈/리뷰/이슈 중 하나")
    summary: str        = Field(description="한 줄 요약 (30자 이내)")
    brand_score: int    = Field(description="브랜드 기회 점수 1~5", ge=1, le=5)
    campaign_angle: str = Field(description="캠페인 참여 방향 제안 (1~2문장)")

class TrendReport(BaseModel):
    date: str                = Field(description="YYYY-MM-DD")
    category: str            = Field(description="요청 카테고리")
    cards: list[TrendCard]   = Field(description="트렌드 카드 3~5개")
    golden_pick: str         = Field(description="골든타임 트렌드 제목")

print("TrendCard, TrendReport 스키마 정의 완료")

TrendCard, TrendReport 스키마 정의 완료


---
## 4. 캐시 레이어

> 같은 날 같은 카테고리 요청 → API 미호출, 캐시 결과 즉시 반환

In [4]:
import diskcache
from datetime import date

_cache = diskcache.Cache("./cache_store")

def get_or_fetch(category: str, fetch_fn):
    key = f"{date.today()}_{category}"
    if key in _cache:
        print(f"[캐시 히트] {key} — API 호출 없이 반환")
        return _cache[key]
    print(f"[API 호출] {key}")
    result = fetch_fn(category)
    _cache.set(key, result, expire=86400)   # 당일 자정까지 유효
    return result

print("캐시 레이어 준비 완료")

캐시 레이어 준비 완료


---
## 5. Tool 1 — YouTube Data API v3

> 한국 인기 급상승 Shorts 수집 + 카테고리 필터링

In [5]:
from langchain_core.tools import tool
from googleapiclient.discovery import build
from datetime import datetime, timedelta, timezone
import re

# 모듈 레벨 YouTube 클라이언트 — 한 번만 생성해 모든 도구가 공유
youtube = build("youtube", "v3", developerKey=os.getenv("YOUTUBE_API_KEY"))

# [v0.5 수정] 트렌딩 = 최신성 확보:
#   order="viewCount"만 쓰면 '역대 누적 조회수 1위'(수년 전 영상)가 올라온다.
#   publishedAfter로 '최근 N일 업로드분'으로 좁히면 → 그 안에서 조회수순 = 진짜 트렌딩.
RECENT_DAYS = 30   # 좁힐수록 더 최신이지만 결과 수가 줄어 시연 안정성↓ (7~30 권장)

# [v0.6] 한국 콘텐츠 필터:
#   YouTube API에는 '국가/언어 하드 필터'가 없다(regionCode/relevanceLanguage는 힌트일 뿐).
#   그래서 제목+설명의 한글 포함 여부로 후처리해 글로벌(영어권) 영상을 걸러낸다.
_HANGUL = re.compile(r"[가-힣]")
MIN_HANGUL = 2          # 제목+설명에 한글이 이 수 이상이면 한국 콘텐츠로 간주
KR_MIN_RESULTS = 3      # 필터 후 결과가 이보다 적으면 빈 화면 방지를 위해 원본으로 폴백


def _is_korean(row: dict, min_hangul: int = MIN_HANGUL) -> bool:
    """제목+설명에 한글이 min_hangul자 이상이면 한국 콘텐츠로 본다."""
    text = f"{row.get('title', '')} {row.get('description', '')}"
    return len(_HANGUL.findall(text)) >= min_hangul


def _recent_published_after(days: int = RECENT_DAYS) -> str:
    """YouTube API용 RFC3339(UTC) 시각 문자열. 이 시각 이후 업로드분만 검색한다."""
    return (datetime.now(timezone.utc) - timedelta(days=days)).strftime("%Y-%m-%dT%H:%M:%SZ")


def _iso8601_to_seconds(dur: str) -> int:
    """ISO8601 길이(PT#H#M#S)를 초로 변환"""
    m = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", dur or "")
    if not m:
        return 0
    h, mi, s = (int(x) if x else 0 for x in m.groups())
    return h * 3600 + mi * 60 + s

def _fetch_shorts(category: str) -> list[dict]:
    """카테고리 키워드로 검색 → 실제 Shorts(≤3분)만 선별 → 조회수순 정렬.
    mostPopular(전체 인기) 대신 카테고리 검색을 써서 '브랜드 관련성'을 확보한다.
    [v0.5] publishedAfter로 최근 RECENT_DAYS일 업로드분만 → '트렌딩(최신성)' 보장.
    [v0.6] 한글 후처리 필터로 한국 콘텐츠만 남긴다."""
    keywords = CATEGORY_KEYWORDS.get(category, [category])[:3]   # 쿼터 절약: 상위 3개 키워드
    published_after = _recent_published_after()                  # 최근 N일 이내만

    # 1) 키워드별 검색으로 후보 video_id 수집 (search.list = 100 unit/회)
    candidate_ids = []
    for kw in keywords:
        try:
            sr = youtube.search().list(
                part="snippet", q=f"{kw} shorts", type="video",
                videoDuration="short", order="viewCount",
                regionCode="KR", relevanceLanguage="ko", maxResults=8,
                publishedAfter=published_after,   # ← [v0.5] 최신성: 최근 N일 업로드분만
            ).execute()
            candidate_ids += [it["id"]["videoId"] for it in sr.get("items", [])]
        except Exception as e:
            print(f"  [검색 실패] {kw}: {e}")
    candidate_ids = list(dict.fromkeys(candidate_ids))   # 중복 제거(순서 유지)
    if not candidate_ids:
        return []

    # 2) 상세 정보 일괄 조회 (videos.list = 1 unit) — 길이/조회수 확인용
    det = youtube.videos().list(
        part="snippet,contentDetails,statistics",
        id=",".join(candidate_ids[:50]),
    ).execute()

    results = []
    for item in det.get("items", []):
        secs = _iso8601_to_seconds(item.get("contentDetails", {}).get("duration", ""))
        if secs == 0 or secs > 180:       # Shorts는 최대 3분 — 그 이상은 제외
            continue
        sn = item["snippet"]
        results.append({
            "video_id":     item["id"],
            "title":        sn.get("title", ""),
            "description":  sn.get("description", "")[:300],
            "tags":         sn.get("tags", [])[:10],
            "views":        int(item.get("statistics", {}).get("viewCount", 0)),
            "published_at": sn.get("publishedAt", ""),   # ← [v0.5] 업로드 시각(최신성 검증용)
            "duration_s":   secs,
            "is_short60":   secs <= 60,
        })

    # 2-1) [v0.6] 한국 콘텐츠만 남긴다. 단, 너무 적게 남으면 빈 화면 방지를 위해 원본으로 폴백.
    korean = [r for r in results if _is_korean(r)]
    if len(korean) >= KR_MIN_RESULTS:
        results = korean
    else:
        print(f"  [경고] 한국 콘텐츠가 {len(korean)}개뿐 → 전체 결과로 폴백(키워드를 더 한국적으로 조정 권장)")

    # 3) 60초 이하 우선 + 조회수순 정렬 후 상위 10개
    results.sort(key=lambda r: (r["is_short60"], r["views"]), reverse=True)
    return results[:10]


@tool
def get_trending_shorts(category: str) -> list[dict]:
    """한국 유튜브에서 카테고리 관련 인기 Shorts(≤3분)를 수집합니다.

    Args:
        category: 브랜드 카테고리 (뷰티, 식음료, 패션, 테크, 피트니스)
    """
    return get_or_fetch(category, _fetch_shorts)


print(f"get_trending_shorts tool 준비 완료 (최근 {RECENT_DAYS}일 + 조회수순 + 한국 콘텐츠 필터)")


get_trending_shorts tool 준비 완료 (최근 30일 + 조회수순 + 한국 콘텐츠 필터)


---
## 6. Tool 2 — 자막 추출

> YoutubeLoader로 자막 추출. 자막 없으면 제목+설명으로 fallback

In [6]:
from langchain_community.document_loaders import YoutubeLoader

@tool
def get_transcript(video_id: str, title: str = "", description: str = "") -> str:
    """유튜브 영상의 자막을 추출합니다. 자막 없으면 제목+설명을 반환합니다.

    Args:
        video_id:    YouTube 영상 ID
        title:       자막 없을 때 fallback용 제목
        description: 자막 없을 때 fallback용 설명
    """
    try:
        loader = YoutubeLoader.from_youtube_url(
            f"https://www.youtube.com/watch?v={video_id}",
            language=["ko", "en"],
        )
        docs = loader.load()
        if not docs or not docs[0].page_content.strip():
            raise ValueError("empty transcript")
        return docs[0].page_content[:2000]
    except Exception:
        fallback = f"[자막없음] {title}"
        if description:
            fallback += f" / {description[:200]}"
        return fallback


print("get_transcript tool 준비 완료")

get_transcript tool 준비 완료


---
## 7. Tool 3 — YouTube 키워드 검색

> 특정 트렌드 심층 탐색 시 사용 — 시나리오 B에서 에이전트가 자동 호출

In [7]:
@tool
def search_shorts(query: str) -> str:
    """주어진 키워드로 유튜브 Shorts를 추가 검색합니다.
    인기 트렌드 목록 밖의 영상을 심층 탐색할 때 사용합니다.
    영상 제목 / 채널 / video_id 목록을 반환합니다."""
    try:
        # 모듈 레벨에서 만든 youtube 클라이언트를 그대로 사용
        response = youtube.search().list(
            part="snippet",
            q=query + " shorts",
            type="video",
            videoDuration="short",   # 짧은 영상 우선
            maxResults=5,
            relevanceLanguage="ko",
            regionCode="KR",
            order="viewCount",
            publishedAfter=_recent_published_after(),   # ← [v0.5] 최신성: 최근 N일 업로드분만(트렌딩)
        ).execute()

        results = []
        for item in response.get("items", []):
            video_id = item["id"]["videoId"]
            title    = item["snippet"]["title"]
            channel  = item["snippet"]["channelTitle"]
            published = item["snippet"].get("publishedAt", "")[:10]
            results.append(f"- {title} (채널: {channel}, 업로드: {published}, ID: {video_id})")

        return "\n".join(results) if results else "검색 결과가 없습니다."
    except Exception as e:
        return f"검색 중 오류 발생: {str(e)}"


print("search_shorts tool 준비 완료")


search_shorts tool 준비 완료


---
## 7. 에이전트 구성

> `create_agent` (LangChain 1.x)

In [8]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain.agents import create_agent
import ast

# ── LLM 선택 ───────────────────────────────────────────────────────────────
# [옵션 1] Google Gemini (기본값)
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

# [옵션 2] OpenAI — 위 두 줄 주석 처리 후 아래 두 줄 활성화
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o-mini")
# ───────────────────────────────────────────────────────────────────────────

tools = [
    get_trending_shorts,   # Tool 1: 인기 급상승 Shorts 수집 (캐시 포함)
    get_transcript,        # Tool 2: 영상 자막 추출
    search_shorts,         # Tool 3: 키워드로 Shorts 추가 검색 (심층 탐색)
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

def extract_text(msg) -> str:
    """
    LangGraph + Gemini 환경에서 content가 올 수 있는 모든 형태 처리:
    - str (일반 텍스트)
    - str (리스트처럼 생긴 문자열: LangGraph 직렬화 결과)
    - list[dict]
    - list[Pydantic 객체]
    """
    content = msg.content

    # 1) 문자열인 경우
    if isinstance(content, str):
        stripped = content.strip()
        # 리스트 형태 문자열이면 파싱 시도 (LangGraph 직렬화 케이스)
        if stripped.startswith("[") and stripped.endswith("]"):
            try:
                parsed = ast.literal_eval(stripped)
                if isinstance(parsed, list):
                    content = parsed   # 아래 리스트 처리로 넘김
                else:
                    return content
            except Exception:
                return content
        else:
            return content  # 일반 텍스트 문자열

    # 2) 리스트인 경우
    if isinstance(content, list):
        parts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                parts.append(str(block.get("text", "")))
            elif hasattr(block, "type") and getattr(block, "type") == "text":
                parts.append(str(getattr(block, "text", "")))
        if parts:
            return "".join(parts)

    return str(content)

chat_history = []
print(f"에이전트 준비 완료 — 모델: {llm.model if hasattr(llm, 'model') else llm.model_name}")
print(f"등록된 도구: {[t.name for t in tools]}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


에이전트 준비 완료 — 모델: gemini-3.1-flash-lite
등록된 도구: ['get_trending_shorts', 'get_transcript', 'search_shorts']


In [9]:
# ── 사전 검증 (문제 발생 시 여기서 먼저 확인) ──────────────────────────────

# 1) YouTube API 직접 호출 테스트
sample = _fetch_shorts("뷰티")
print(f"[Tool 검증] 수집된 영상 수: {len(sample)}개")
if sample:
    print(f"  첫 번째 영상: {sample[0]['title']}")

# 2) 에이전트 응답 포맷 확인
_test = agent.invoke({"messages": [HumanMessage(content="안녕")]})
_msg  = _test["messages"][-1]
_ct   = _msg.content
print(f"\n[포맷 검증] content 타입: {type(_ct).__name__}", end="")
if isinstance(_ct, list):
    print(f" / 첫 번째 블록 타입: {type(_ct[0]).__name__}")
else:
    print()

# 3) extract_text 동작 확인
print(f"[추출 검증] extract_text 결과: {extract_text(_msg)[:50]}...")

[Tool 검증] 수집된 영상 수: 10개
  첫 번째 영상: 장원영 메이크업의 필수품 #kpop #shorts

[포맷 검증] content 타입: list / 첫 번째 블록 타입: dict
[추출 검증] extract_text 결과: 안녕하세요! 브랜드 마케터 전용 유튜브 트렌드 분석 AI입니다.

최신 유튜브 Shorts...


---
## 8. 데모 실행

> 아래 3개 셀을 **순서대로** 실행하세요. 대화 맥락이 이어집니다.

In [9]:
# 이전 실행에서 오류 결과가 캐시됐을 경우 초기화 (필요시 실행)
_cache.clear()
print("캐시 초기화 완료")

캐시 초기화 완료


### 시나리오 A — 아침 트렌드 브리핑

In [10]:
user_input = "오늘 뷰티 관련 유튜브 쇼츠 트렌드 알려줘"
print(f"사용자: {user_input}\n")

messages = chat_history + [HumanMessage(content=user_input)]
result   = agent.invoke({"messages": messages})
output   = extract_text(result["messages"][-1])

print(f"\n에이전트:\n{output}")
chat_history = result["messages"]

사용자: 오늘 뷰티 관련 유튜브 쇼츠 트렌드 알려줘

[캐시 히트] 2026-07-01_뷰티 — API 호출 없이 반환

에이전트:
오늘의 뷰티 유튜브 쇼츠 트렌드를 분석한 결과입니다. 마케팅 캠페인 전략 수립에 참고하시기 바랍니다.

### 뷰티 트렌드 카드

| 제목 | 카테고리 | 한 줄 요약 | 기회 점수 | 캠페인 방향 |
| :--- | :--- | :--- | :---: | :--- |
| **아이돌 메이크업 "필수템" 공개** | 리뷰/정보 | 장원영, 원희 등 아이돌의 데일리 립/메이크업 조합 열풍 | **4점** | 인플루언서와 함께하는 '아이돌 무드' 립 제품 챌린지 |
| **청담샵 비법 + 가성비(다이소) 조합** | 정보/팁 | 전문가의 기술에 저렴한 뷰티템을 더한 커버 메이크업 | **4점** | 고급스러운 비주얼의 룩북과 실제 사용한 가성비 제품 매칭 |
| **여름 대비 "지속력" 끝판왕 검증** | 리뷰/비교 | 파운데이션/아이라이너 등 여름 맞춤형 지속력 테스트 | **5점** | 브랜드 제품의 압도적인 지속력을 보여주는 '12시간 생존' 챌린지 |

---

### [골든타임 트렌드] 추천: "아이돌 데일리 립 조합 (Idol Lip Combo)"
*   **추천 이유:** 
    현재 뷰티 쇼츠의 핵심은 **'따라 하고 싶은 아이돌의 무드'**와 **'검증된 지속력'** 두 가지로 압축됩니다. 특히 아이돌이 직접 사용했거나, 샵에서 사용한 제품이라는 사실만으로도 구매 전환이 빠르게 일어납니다. 
*   **마케팅 제언:** 
    단순 제품 홍보를 넘어, 고객이 해당 제품을 구매해야 하는 '상황(예: 여름 페스티벌, 데이트)'을 명확히 설정하고, 15초 이내에 드라마틱한 비포/애프터와 지속력을 보여주는 것이 가장 효과적입니다. 현재 '지속력'에 대한 사용자 니즈가 매우 높으므로, 워터프루프나 고정력을 강조한 캠페인을 즉시 런칭하시길 권장합니다.


#### 🔎 시나리오 A 검증 — 카드가 실제 유튜브 영상에 근거했는지 확인

위 트렌드 카드(`"이 신발 얼마게?" 챌린지` 등)는 **LLM이 가공·요약한 트렌드 이름**이라, 그 자체로는 유튜브에서 검색해도 똑같이 나오지 않습니다. 검증 대상은 도구가 LLM에게 넘긴 **raw 데이터(video_id·제목·조회수)**입니다.

아래 셀은 한 번에 3가지를 확인합니다.
- **검증 2 (그라운딩)**: 에이전트가 `get_trending_shorts`를 실제로 호출해 그 데이터를 받았는가 ← 가장 중요
- **검증 1 (실재 URL)**: 받은 video_id로 만든 유튜브 링크 — 클릭하면 실제 영상이 열림
- **검증 3 (ID 실존 교차검증)**: 그 video_id들을 API로 재조회해 살아있는 영상인지 일괄 확인

In [11]:
# ── 시나리오 A 검증: 그라운딩 + 실재 URL + ID 실존 교차검증 ──────────────────
from langchain_core.messages import ToolMessage, AIMessage
import json, ast


def _parse_tool_rows(tool_msg) -> list[dict]:
    """ToolMessage.content를 list[dict](영상 row 목록)로 복원한다.
    LangChain은 도구 반환값을 JSON 문자열(true/false/null)로 직렬화하는 경우가 많아
    json.loads를 먼저 시도하고, 파이썬 repr(True/False/None)이면 ast.literal_eval로 보완한다."""
    content = tool_msg.content

    # 1) 이미 리스트인 경우
    if isinstance(content, list):
        if content and isinstance(content[0], dict) and "video_id" in content[0]:
            return content   # row 리스트
        # content block 리스트([{'type':'text','text':'...'}])면 텍스트만 이어붙인다
        content = "".join(
            b.get("text", "") for b in content
            if isinstance(b, dict) and b.get("type") == "text"
        )

    if not isinstance(content, str) or not content.strip():
        return []

    # 2) JSON → 파이썬 repr 순으로 파싱 시도
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(content)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            continue
    return []


# ── 검증 2 (그라운딩): 에이전트가 get_trending_shorts를 실제로 호출했는가 ──
tool_calls = [
    (tc["name"], tc.get("args"))
    for m in chat_history if isinstance(m, AIMessage)
    for tc in (m.tool_calls or [])
]
tms = [m for m in chat_history
       if isinstance(m, ToolMessage) and m.name == "get_trending_shorts"]

print("── 검증 2: 그라운딩 ──")
print("도구 호출 내역:", tool_calls or "(없음)")
if not tms:
    print("❌ get_trending_shorts 미호출 → 카드가 실제 데이터 근거가 아닐 수 있음(환각 위험). 검증 중단.")
    rows = []
else:
    print(f"✅ get_trending_shorts 호출 확인 — ToolMessage {len(tms)}건 (에이전트가 실제 데이터를 받음)")
    rows = _parse_tool_rows(tms[-1])
    if not rows:
        # ToolMessage 파싱 실패 시: 호출 인자의 category로 캐시에서 직접 로드 (데이터는 분명히 존재)
        cat = next((tc.get("args", {}).get("category")
                    for m in chat_history if isinstance(m, AIMessage)
                    for tc in (m.tool_calls or []) if tc["name"] == "get_trending_shorts"), None)
        print(f"   ⚠ ToolMessage 파싱 실패 → 캐시에서 직접 로드 (category={cat})")
        if cat:
            rows = _cache.get(f"{date.today()}_{cat}", []) or []

# ── 검증 1 (실재 URL + 최신성): video_id로 실제 유튜브 링크 + 업로드 날짜 ──
print(f"\n── 검증 1: 실제 영상 URL ({len(rows)}개) ──")
for r in rows:
    _d = (r.get("published_at") or "")[:10]
    print(f"[{r.get('views', 0):,}회 | 업로드 {_d or '?'}] {r.get('title', '')}")
    print(f"   https://www.youtube.com/watch?v={r.get('video_id')}")

# ── 검증 3 (ID 실존): video_id를 API로 재조회해 실존 여부 일괄 확인 ──
ids = [r.get("video_id") for r in rows if r.get("video_id")]
if ids:
    resp = youtube.videos().list(part="id", id=",".join(ids)).execute()
    returned = {it["id"] for it in resp.get("items", [])}   # 존재하지 않는 ID는 응답에서 빠짐
    print("\n── 검증 3: video_id 실존 교차검증 ──")
    for vid in ids:
        print(f"{'✅ 실존' if vid in returned else '❌ 없음(삭제/비공개)'}  {vid}")
    print(f"결과: {sum(v in returned for v in ids)}/{len(ids)}개 실존")
else:
    print("\n── 검증 3 생략: video_id 없음 ──")

── 검증 2: 그라운딩 ──
도구 호출 내역: [('get_trending_shorts', {'category': '뷰티'})]
✅ get_trending_shorts 호출 확인 — ToolMessage 1건 (에이전트가 실제 데이터를 받음)

── 검증 1: 실제 영상 URL (10개) ──
[3,798,121회 | 업로드 2026-06-22] 장원영 메이크업의 필수품 #kpop #shorts
   https://www.youtube.com/watch?v=OnOb0b6D12E
[67,325회 | 업로드 2026-06-13] 원희가 공개한 '데일리 립 조합' 3가지
   https://www.youtube.com/watch?v=6Ulsrgfdoyc
[28,459회 | 업로드 2026-06-01] 데일리 메이크업에 딱 맞는 글로시 & 너리쉬 핑크 립틴트 💓 #숏츠 #바이럴 #뷰티 #두인 #메이크업 #정설
   https://www.youtube.com/watch?v=rVm6aR9U0gc
[9,339회 | 업로드 2026-06-16] 지속력 미친 🔥절대 안 번지는 아이라이너 정체
   https://www.youtube.com/watch?v=D63Uf3RIlfE
[9,278회 | 업로드 2026-06-09] 3살 리엘이의 화장법 #국제가족 #국제부부 #캐나다리리남매 #shorts
   https://www.youtube.com/watch?v=yQmBv2R1kD0
[7,097회 | 업로드 2026-06-01] 🥊헤라VS에스티로더 파운데이션 지속력 대결 #SHORTS
   https://www.youtube.com/watch?v=oychukdgLSw
[6,271회 | 업로드 2026-06-06] 젤리 핑크 글로시 립틴트로 데일리 메이크업 완성 🍬 #shorts #viral #뷰티 #douyin #메이크업 #정설
   https://www.youtube.com/watch?v=-NrET1AbEJQ
[3,827회 | 업로드 2026-06-08] 청담샵에서 배워온 제니메이

### 시나리오 B — 특정 트렌드 심층 탐색 (Tool 3 활용)

In [12]:
# [v0.5 보완] search_shorts 도구 사용을 명시적으로 강제하고, 결과를 video_id와 함께 인용하도록 요청
#   - 기존: "직접 검색해서" 만으로는 모델이 도구를 건너뛰고 일반 지식으로 답하는 경우가 있었음
#   - 변경: 도구명을 직접 지목 + video_id 명시 + 추측 금지 → 실제 검색 결과에 근거하도록 유도
user_input = (
    "첫 번째 트렌드 키워드로 search_shorts 도구를 반드시 호출해서 실제 유튜브 영상을 더 찾아줘. "
    "찾은 영상은 제목과 video_id를 그대로 인용하고, 그 영상들만 근거로 심층 분석해줘. "
    "search_shorts가 반환한 목록에 있는 video_id만 인용하고, 그 목록에 없는 ID는 절대 만들어내지 마. "
    "검색 결과에 없는 내용은 추측하지 말고, 없으면 없다고 말해줘."
)
print(f"사용자: {user_input}\n")

messages = chat_history + [HumanMessage(content=user_input)]
result   = agent.invoke({"messages": messages})
output   = extract_text(result["messages"][-1])

print(f"\n에이전트:\n{output}")
chat_history = result["messages"]

사용자: 첫 번째 트렌드 키워드로 search_shorts 도구를 반드시 호출해서 실제 유튜브 영상을 더 찾아줘. 찾은 영상은 제목과 video_id를 그대로 인용하고, 그 영상들만 근거로 심층 분석해줘. search_shorts가 반환한 목록에 있는 video_id만 인용하고, 그 목록에 없는 ID는 절대 만들어내지 마. 검색 결과에 없는 내용은 추측하지 말고, 없으면 없다고 말해줘.


에이전트:
검색된 영상을 바탕으로 '아이돌 메이크업 필수템' 키워드에 대한 심층 분석을 진행했습니다.

### 아이돌 메이크업 필수템 심층 분석

| 제목 | video_id | 핵심 요약 |
| :--- | :--- | :--- |
| 아이라인, 눈썹, 입술까지? 청담동 연예인들 이 쓰는 미친 픽서의 정체 | vGbVfXPj5Os | 아이돌 무대 메이크업의 핵심은 '고정력'이며, 전문가(청담샵)가 사용하는 픽서 제품이 트렌드임 |
| 태연도 인정한 르세라핌 은채 인생템?! | nOBSJN2U8TI | 아이돌 간의 추천(인생템) 연결 고리가 팬덤 소비를 자극함 |
| 한여름 무대 조명 아래서도 하루 종일 뽀송함, 여돌 워터파크 탈출기✨ | V2V_b-gtw3Q | 무대 조명과 워터파크라는 극한 상황을 제시하여 제품의 '지속력'과 '뽀송함'을 증명함 |
| 소유가 인정한 아이돌 여름 필수 혁신템! | txlg1XsfxEQ | '혁신템'이라는 단어를 사용하여 단순 화장품을 넘어선 여름철 아이돌 필수 관리 루틴을 강조함 |

---

### 마케팅 시사점 및 전략 제안

1.  **"아이돌 = 전문가" 프레임 활용 (vGbVfXPj5Os):** 
    소비자들은 단순 아이돌의 애용품을 넘어, '청담동 샵에서 전문가가 직접 사용하는 제품'에 더 높은 신뢰를 보입니다. 
    *   **전략:** 제품 상세 페이지나 쇼츠에서 '아이돌 담당 메이크업 아티스트가 직접 쓰는'이라는 포인트를 강조하십시오.

2.  **극한 상황 시각화 (V2V_b-gtw3Q):** 
    단순 발색 

#### 🔎 시나리오 B 검증 — `search_shorts`가 실제로 호출됐는지 확인

시나리오 B의 "심층 분석"이 **실제 검색 결과**에 근거한 것인지, 아니면 모델이 도구를 건너뛰고 **일반 지식으로 지어낸 것**인지 확인합니다. 핵심은 `search_shorts`라는 이름의 `ToolMessage`가 대화 기록에 실제로 존재하는가입니다.

- ✅ ToolMessage 존재 → 검색 결과(제목·채널·video_id)에 근거한 분석
- ❌ ToolMessage 없음 → 검색 없이 답한 환각 가능성 (이 경우 v0.5에서 강화한 프롬프트로 다시 실행 권장)

In [13]:
# ── 시나리오 B 검증 (강화판): search_shorts 호출 확인 + '인용 ID 그라운딩' 자동 점검 ──
from langchain_core.messages import ToolMessage, AIMessage
import re, json, ast

# 1) search_shorts 실제 호출 여부 + 반환 영상
b_calls = [
    tc.get("args")
    for m in chat_history if isinstance(m, AIMessage)
    for tc in (m.tool_calls or []) if tc["name"] == "search_shorts"
]
b_tms = [m for m in chat_history
         if isinstance(m, ToolMessage) and m.name == "search_shorts"]

print("search_shorts 호출 인자:", b_calls or "(없음)")
if not b_tms:
    print("\n❌ search_shorts 미호출 → '심층 분석'이 실제 검색이 아닌 환각일 수 있습니다.")
    print("   시나리오 B 셀을 다시 실행하세요.")
else:
    print(f"\n✅ search_shorts 호출 확인 — {len(b_tms)}건")
    print("─ 검색이 반환한 실제 영상 목록(제목 / 채널 / video_id) ─")
    print(str(b_tms[-1].content)[:1500])

# 2) [신규] 그라운딩 자가점검:
#    답변이 인용한 video_id가 '도구가 실제 반환한 ID 집합' 안에 있는지 대조.
#    도구 결과에 없는 ID를 인용하면 환각으로 표시한다. (예: PG1gpEdtQv8 류 자동 적발)
def _grounded_ids(history) -> set:
    """대화 중 모든 도구가 실제 반환한 video_id 집합 (get_trending_shorts + search_shorts)."""
    ids = set()
    for m in history:
        if not isinstance(m, ToolMessage):
            continue
        c = m.content
        if isinstance(c, list):
            c = "".join(b.get("text", "") for b in c
                        if isinstance(b, dict) and b.get("type") == "text")
        text = c if isinstance(c, str) else str(c)
        ids |= set(re.findall(r"video_id['\"]?\s*[:=]\s*['\"]?([A-Za-z0-9_-]{11})", text))  # Tool 1
        ids |= set(re.findall(r"\bID:\s*([A-Za-z0-9_-]{11})", text))                        # Tool 3
    return ids

def _cited_ids(text: str) -> set:
    """답변 본문에서 인용된 11자 video_id: 'ID: xxx' / 'v=xxx' / `xxx`(백틱) 모두."""
    text = text or ""
    ids  = set(re.findall(r"\bID\s*[:=]\s*([A-Za-z0-9_-]{11})", text))
    ids |= set(re.findall(r"v=([A-Za-z0-9_-]{11})", text))
    ids |= set(re.findall(r"`([A-Za-z0-9_-]{11})`", text))   # 표 안 백틱 인용 대응
    return ids

final_ai = next((m for m in reversed(chat_history)
                 if isinstance(m, AIMessage) and not (m.tool_calls or [])), None)
answer_text = extract_text(final_ai) if final_ai else ""

grounded   = _grounded_ids(chat_history)
cited      = _cited_ids(answer_text)
ungrounded = cited - grounded

print("\n── 인용 ID 그라운딩 자가점검 ──")
print(f"답변이 인용한 ID: {sorted(cited) or '(없음)'}")
if not cited:
    print("⚠️  답변에 video_id 인용이 없습니다 — 도구 결과 인용을 요청하세요.")
elif ungrounded:
    print(f"❌ 도구가 반환한 적 없는 ID: {sorted(ungrounded)} → 환각 가능성. 시나리오 B 재실행 권장.")
else:
    print(f"✅ 인용 ID 전부가 실제 도구 반환 결과에 존재 ({len(cited)}/{len(cited)})")

search_shorts 호출 인자: [{'query': '아이돌 메이크업 필수템'}]

✅ search_shorts 호출 확인 — 1건
─ 검색이 반환한 실제 영상 목록(제목 / 채널 / video_id) ─
- 땀냄새 용납 못하는 아이돌 필수템 (채널: 언니픽, 업로드: 2026-06-12, ID: VGoAiFEyfgY)
- 아이라인, 눈썹, 입술까지? 청담동 연예인들 이 쓰는 미친 픽서의 정체 #메이크업픽서 #권은비픽서 #여름메이크업 (채널: 셀럽콕, 업로드: 2026-06-21, ID: vGbVfXPj5Os)
- 태연도 인정한 르세라핌 은채 인생템?! (채널: FOUND BY 파운드바이, 업로드: 2026-06-30, ID: nOBSJN2U8TI)
- 한여름 무대 조명 아래서도 하루 종일 뽀송함, 여돌 워터파크 탈출기✨ #전소미 #솔라 #소유 #아이돌추천#shorts (채널: 스타꿀템, 업로드: 2026-06-12, ID: V2V_b-gtw3Q)
- (광고)소유가 인정한 아이돌 여름 필수 혁신템! #short #여름템 #여름필수템 #여름 #데오드란트 #샤워티슈 (채널: 이거뭐지연구소, 업로드: 2026-06-24, ID: txlg1XsfxEQ)

── 인용 ID 그라운딩 자가점검 ──
답변이 인용한 ID: (없음)
⚠️  답변에 video_id 인용이 없습니다 — 도구 결과 인용을 요청하세요.


### 시나리오 C — 브랜드 적합성 판단

In [14]:
user_input = "우리 슬로건이 '보송보송한 피부'인데 이 트렌드랑 어울려?"
print(f"사용자: {user_input}\n")

messages = chat_history + [HumanMessage(content=user_input)]
result   = agent.invoke({"messages": messages})
output   = extract_text(result["messages"][-1])

print(f"\n에이전트:\n{output}")
chat_history = result["messages"]

사용자: 우리 슬로건이 '보송보송한 피부'인데 이 트렌드랑 어울려?


에이전트:
네, 브랜드 슬로건인 **'보송보송한 피부'는 현재 유튜브 뷰티 쇼츠 트렌드와 매우 강력하게 부합**합니다.

지금의 뷰티 트렌드는 단순한 '색조'를 넘어 **'여름철 무너지지 않는 완벽한 베이스'**를 지향하고 있기 때문입니다. 특히 위에서 분석한 영상들(특히 `V2V_b-gtw3Q`)을 보면, 여름 시즌 소비자들은 **"하루 종일 뽀송함"**을 갈구하고 있습니다.

귀사의 슬로건을 트렌드에 맞춰 어떻게 녹여낼지 전략적으로 제안해 드립니다.

### '보송보송한 피부' 브랜드 캠페인 기회 (점수: 5점)

**1. 트렌드와의 연결 고리: [무너짐 제로 챌린지]**
*   **분석 근거:** `V2V_b-gtw3Q`(한여름 무대 조명 아래서도 뽀송함) 영상에서 보듯, 소비자들은 덥고 습한 환경에서도 보송함을 유지하는 제품에 반응합니다.
*   **캠페인 방향:** "한여름 35도 야외에서도 뽀송할 수 있을까?"라는 주제로, 귀사의 제품을 사용한 후 6시간, 12시간 뒤의 피부 상태를 체크하는 **'보송보송 타임랩스'** 콘텐츠를 제작하세요.

**2. 전문가 신뢰도 구축: [청담샵 베이스 공식]**
*   **분석 근거:** `vGbVfXPj5Os`(청담동 연예인 픽서) 영상처럼, 소비자들은 전문가의 베이스 메이크업 비법에 열광합니다.
*   **캠페인 방향:** 귀사의 제품이 단순히 '보송하다'는 홍보를 넘어, 메이크업 아티스트를 섭외하여 **"청담샵에서 아이돌 베이스를 마무리할 때 반드시 사용하는 '보송보송 비결'"**로 포지셔닝하세요.

**3. 소셜 증명: [보송보송 챌린지]**
*   **분석 근거:** `nOBSJN2U8TI`(태연도 인정한 인생템)와 같이 아이돌의 인증은 곧 강력한 구매 명분이 됩니다.
*   **캠페인 방향:** 인플루언서와 함께 "하루 종일 뽀송했나요?"라는 **질문형 챌린지**를 진행하세요. 틱톡이나 쇼츠에서 유행하는 '기름종이 테스트'를 활용하

---
## 9. Structured Output

> 위 대화 결과를 Pydantic 스키마(TrendReport)로 구조화

In [15]:
from datetime import date as dt

structured_llm = llm.with_structured_output(TrendReport)

# tool_call이 아닌 '실제 응답' AI 메시지 중 마지막(=최종 리포트)을 사용
report_msgs = [
    m for m in chat_history
    if isinstance(m, AIMessage)
    and not getattr(m, "tool_calls", None)
    and extract_text(m).strip()
]
agent_response = extract_text(report_msgs[-1]) if report_msgs else ""

prompt_text = f"""
아래는 오늘({dt.today()}) 분석한 한국 유튜브 Shorts 트렌드 결과입니다.
이를 TrendReport 형식으로 구조화하세요.

규칙:
- category 필드에는 분석 결과에 실제로 나타난 카테고리를 그대로 넣으세요.
- 데이터에 없는 내용을 지어내지 마세요.

분석 결과:
{agent_response}
"""

report = structured_llm.invoke(prompt_text)
print("=== Structured Output (TrendReport) ===")
print(report.model_dump_json(indent=2, ensure_ascii=False))


=== Structured Output (TrendReport) ===
{
  "date": "2026-07-01",
  "category": "뷰티 쇼츠 트렌드",
  "cards": [
    {
      "title": "무너짐 제로 챌린지",
      "category": "챌린지",
      "summary": "덥고 습한 날씨 속 제품 사용 후 시간별 피부 상태 체크",
      "brand_score": 5,
      "campaign_angle": "한여름 35도 야외에서 제품 사용 후 6~12시간 뒤의 피부 상태를 보여주는 타임랩스 콘텐츠를 제작하세요."
    },
    {
      "title": "청담샵 베이스 공식",
      "category": "이슈",
      "summary": "전문가 메이크업 아티스트가 전하는 베이스 비결",
      "brand_score": 4,
      "campaign_angle": "메이크업 아티스트를 섭외하여 아이돌 베이스 메이크업의 핵심으로 제품을 포지셔닝하세요."
    },
    {
      "title": "보송보송 챌린지",
      "category": "챌린지",
      "summary": "인플루언서와 함께하는 기름종이 테스트 챌린지",
      "brand_score": 5,
      "campaign_angle": "기름종이 테스트를 활용해 전후 변화를 시각적으로 보여주고 소비자 참여형 챌린지를 유도하세요."
    }
  ],
  "golden_pick": "무너짐 제로 챌린지"
}


In [ ]:
# ── (선택) 자유 대화 루프 — 채점/시연 흐름과 분리된 '수동 실행 전용' ──────────────
#   ⚠️ 'Run All' 시 input()에서 멈추므로 노트북 맨 끝에 두고 필요할 때만 수동 실행.
#   ⚠️ A·B·C의 chat_history를 오염시키지 않도록 '복사본'에서만 대화한다.
print("자유 대화를 시작합니다. 종료하려면 'q' 또는 '종료'를 입력하세요.\n")

_local_history = list(chat_history)   # A·B·C 결과 복사 — 원본 보존
while True:
    user_input = input("사용자: ").strip()
    if not user_input:
        continue
    if user_input.lower() in ("q", "종료", "exit", "quit"):
        print("대화를 종료합니다.")
        break
    messages = _local_history + [HumanMessage(content=user_input)]
    result   = agent.invoke({"messages": messages})
    print(f"\n에이전트:\n{extract_text(result['messages'][-1])}\n")
    print("-" * 50)
    _local_history = result["messages"]   # 복사본만 갱신 (전역 chat_history 불변)

자유 대화를 시작합니다. 종료하려면 'q' 또는 '종료'를 입력하세요.



사용자:  여백 삭제 기술로 아이돌 무드를 완성하고 빛나는 하루를 시작함. 이런 키워드가 뷰티 트렌드라는데 어떤 메이크업 아이템이랑 잘 맞아?



에이전트:
'여백 삭제'와 '빛나는 하루'라는 키워드는 최근 뷰티 트렌드인 **'입체감'**과 **'광채(글로우)'** 메이크업과 완벽하게 맞닿아 있습니다.

아이돌의 여백을 삭제하고 얼굴을 빛나게 만드는 메이크업 아이템 4가지를 추천해 드립니다. 브랜드의 제품군과 가장 잘 맞는 아이템을 선택해 보세요.

---

### 1. 여백 삭제의 핵심: [하이라이터 & 쉐딩 팔레트]
'여백 삭제'는 결국 얼굴의 명암 대비를 통해 면적을 좁아 보이게 만드는 것이 핵심입니다.
*   **왜 잘 맞는가?:** 광대나 턱의 여백을 쉐딩으로 깎고, 얼굴 중심부를 하이라이터로 밝혀 입체감을 주면 얼굴이 훨씬 작고 또렷해 보입니다. 
*   **캠페인 문구:** "여백은 지우고 광채는 채우고, 오늘 하루 빛나는 나를 만나다."

### 2. 빛나는 하루의 완성: [글로우 쿠션 & 톤업 베이스]
'빛나는 하루'라는 슬로건을 시각적으로 가장 직관적으로 보여주는 아이템입니다.
*   **왜 잘 맞는가?:** 잡티를 가리는 것보다 '결광'을 살리는 베이스 메이크업이 최근 트렌드(ID: rVm6aR9U0gc 등에서 보이는 글로시한 피부 표현)입니다. 
*   **캠페인 문구:** "빛나는 하루는 결광부터, 빈틈없는 피부로 완성하는 아이돌 무드."

### 3. 디테일한 무드: [애교살 메이커 & 펜슬 라이너]
아이돌 메이크업의 가장 큰 특징은 눈의 가로 길이를 확장해 얼굴의 여백을 물리적으로 줄이는 것입니다.
*   **왜 잘 맞는가?:** 애교살을 강조하면 눈이 커 보이고 얼굴 여백이 자연스럽게 줄어듭니다. (ID: Zln2OXypMi4에서 강조하는 애굣살 포인트)
*   **캠페인 문구:** "한 끗 차이로 빛나는 하루, 애교살 메이크업으로 여백 삭제 완료."

### 4. 시선을 모으는 마침표: [글로시 틴트]
얼굴 전체에 광채를 더해주어 슬로건인 '빛나는' 이미지를 극대화합니다.
*   **왜 잘 맞는가?:** 최근 뷰티 쇼츠에서 '젤리 핑크 글로시 립'은 필수 치트키입니다. (